In [0]:
%run "../00_common/01_environment_config"

In [0]:
%run "../00_common/02_bronze_helpers"

In [0]:
# Databricks notebook source

# ============================================================
# F1 PROJECT - INGEST RACES
# ============================================================
# Purpose:
# Reads races.csv from S3 Landing, adds ingestion metadata,
# and stores the data as a Delta table in the Bronze layer.
# ============================================================


# COMMAND ----------

# Load environment configuration

# MAGIC %run "../00_common/01_environment_config"


# COMMAND ----------

# Load Bronze helper functions

# MAGIC %run "../00_common/02_bronze_helpers"


# COMMAND ----------

# Source and target paths

source_file = f"{s3_root_path}/landing/races.csv"

target_path = f"{s3_root_path}/Bronze/Kalashri/races"

print("Source File :", source_file)
print("Target Path :", target_path)


# COMMAND ----------

# Define schema

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DateType
)

races_schema = StructType([
    StructField("season", IntegerType(), True),
    StructField("round", IntegerType(), True),
    StructField("url", StringType(), True),
    StructField("raceName", StringType(), True),
    StructField("date", DateType(), True),
    StructField("circuitId", StringType(), True)
])


# COMMAND ----------

# Read races.csv from S3 Landing

races_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("mode", "FAILFAST")
    .schema(races_schema)
    .load(source_file)
)

display(races_df)


# COMMAND ----------

# Check schema

races_df.printSchema()


# COMMAND ----------

# Add ingestion metadata

races_final_df = add_ingestion_metadata(races_df)

display(races_final_df)


# COMMAND ----------

# Write to S3 Bronze in Delta format

(
    races_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(target_path)
)


# COMMAND ----------

# Register S3 Delta location as Unity Catalog table

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS
    bizmetric_databricks_learning.default.bronze_races
    USING DELTA
    LOCATION '{target_path}'
""")


# COMMAND ----------

# Validate Bronze table

bronze_races_df = spark.table(
    "bizmetric_databricks_learning.default.bronze_races"
)

display(bronze_races_df)

print("Total Bronze Records:", bronze_races_df.count())

In [0]:
%sql

SELECT *
FROM delta.`s3://bizf1project123/Bronze/Kalashri/races`;